# PRACTICE 1: CLASSIFYING SPAM EMAILS
## PHẦN 1: KHÁM PHÁ DỮ LIỆU EMAIL (EXPLORATORY DATA ANALYSIS - EDA)

> **Mục tiêu:**
> - Khảo sát toàn diện 39.154 email thô CEAS_08, cấu trúc thuộc tính và khuyết thiếu kỹ thuật.
> - Phân tích đơn biến phân bố nhãn, độ dài tiêu đề và top tên miền người gửi.
> - Khảo sát đa biến, phân phối độ dài văn bản, phát hiện ngoại lai IQR và thủ thuật ngụy trang Bayesian Poisoning.
> - Đúc kết định hướng chiến lược tiền xử lý và kỹ thuật đặc trưng cho Notebook 2 (Duy trì nguyên tắc EDA Read-Only 100%).

---
## 0. GIỚI THIỆU BỘ DỮ LIỆU CEAS_08
- **Quy mô thô:** 39.154 email (khoảng 65 MB).
- **Biến mục tiêu:** `label` (0: Ham - Thư thường, 1: Spam - Thư rác).
- **Các thuộc tính:** `sender`, `receiver`, `date`, `subject`, `body`, `label`, `urls`.

---
## Mục 1: Khám phá dữ liệu tổng quan — Cấu trúc, Chất lượng & Đơn biến (Overview EDA)

### 1.1 Khai báo Thư viện & Cấu hình Thực nghiệm
**Mục tiêu:** Nạp thư viện cốt lõi và cố định `RANDOM_SEED = 42` bảo đảm tính tái lập kết quả.

In [ ]:
# Khai báo thư viện & Thiết lập môi trường
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.autolayout'] = True
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# Thêm đường dẫn gốc để nạp các module từ scripts/
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from data.scripts.eda import (
    load_raw_data, inspect_data_structure, get_univariate_stats,
    detect_iqr_outliers, get_top_keywords_comparison,
    analyze_re_prefix_and_duplicates
)
from data.scripts.data_preprocessing import (
    handle_missing_values, clean_text, process_and_save_clean_data
)

print("Môi trường phân tích và các module data.scripts.eda, data.scripts.data_preprocessing sẵn sàng (RANDOM_SEED = 42).")

### 1.2 Khảo sát Cấu trúc & Đánh giá Sơ bộ Chất lượng Dữ liệu
**Mục tiêu:** Nạp dữ liệu thô, kiểm tra quy mô, kiểu dữ liệu, khuyết thiếu và trùng lặp kỹ thuật.

In [ ]:
# Khảo sát cấu trúc & Đánh giá chất lượng dữ liệu thô qua data.scripts.eda
df = load_raw_data()
struct_info = inspect_data_structure(df)

print(f"1. Quy mô dữ liệu thô: {struct_info['n_rows']:,} dòng, {struct_info['n_cols']} cột")
print("\n2. Giá trị khuyết thiếu tổng quát trên các cột:")
print(struct_info['missing_values'])
print(f"\n3. Trùng lặp thô trên toàn bộ 7 cột: {struct_info['exact_duplicates']} dòng")
print(f"4. Trùng lặp thô trên cặp (subject, body): {struct_info['subject_body_duplicates']} dòng")
display(df.head(5))

### 1.3 Phân tích Đơn biến (Univariate Analysis)
**Mục tiêu:** Khảo sát phân bố biến mục tiêu `label`, sự xuất hiện của `urls`, độ dài tiêu đề và top tên miền người gửi.

In [ ]:
# Phân tích đơn biến qua data.scripts.eda và trực quan hóa 4 thuộc tính
uni_stats = get_univariate_stats(df)
print(f"Thống kê đơn biến: Ham={uni_stats['ham_ratio']:.1f}%, Spam={uni_stats['spam_ratio']:.1f}%, Có URL={uni_stats['url_present_ratio']:.1f}%")
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Biến mục tiêu
df['label'].value_counts().plot.pie(
    labels=['Spam (1)', 'Ham (0)'], autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'],
    ax=axes[0, 0], startangle=90, explode=(0.03, 0)
)
axes[0, 0].set_title("Tỷ lệ Spam vs Ham (Dữ liệu thô)", fontweight='bold')
axes[0, 0].set_ylabel('')

# 2. Thuộc tính URLs
sns.countplot(data=df, x='urls', ax=axes[0, 1], palette=['#3498db', '#f39c12'])
axes[0, 1].set_title("Sự hiện diện của liên kết URLs", fontweight='bold')

# 3. Độ dài tiêu đề
subj_len = df['subject'].fillna('').str.len()
sns.histplot(subj_len[subj_len < 120], bins=30, kde=True, ax=axes[1, 0], color='#9b59b6')
axes[1, 0].set_title("Phân phối Độ dài Tiêu đề (< 120 ký tự)", fontweight='bold')

# 4. Top 10 Tên miền người gửi
domains = df['sender'].astype(str).apply(lambda x: x.split('@')[-1].lower() if '@' in str(x) else 'unknown')
domains.value_counts().head(10).plot.barh(ax=axes[1, 1], color='#1abc9c').invert_yaxis()
axes[1, 1].set_title("Top 10 Tên miền Người gửi", fontweight='bold')

plt.tight_layout()
plt.show()

---
## Mục 2: Khám phá dữ liệu chuyên sâu — Đa biến, Ngoại lai & Đúc kết định hướng (In-depth EDA)

### 2.1 Khảo sát Đa biến: So sánh Độ dài Văn bản giữa Thư Rác và Thư Thường
**Mục tiêu:** Sử dụng Boxplot so sánh phân phối độ dài tiêu đề và nội dung theo nhãn.

In [ ]:
# So sánh độ dài văn bản theo nhãn
df['subject_len'] = df['subject'].fillna('').str.len()
df['body_len'] = df['body'].fillna('').str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=df, x='label', y='subject_len', ax=axes[0], palette=['#2ecc71', '#e74c3c'], showfliers=False)
axes[0].set_xticklabels(['Ham (0)', 'Spam (1)'])
axes[0].set_title("Độ dài Tiêu đề theo Nhãn", fontweight='bold')

sns.boxplot(data=df, x='label', y=np.log1p(df['body_len']), ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_xticklabels(['Ham (0)', 'Spam (1)'])
axes[1].set_title("Log(Độ dài Nội dung + 1) theo Nhãn", fontweight='bold')
plt.tight_layout()
plt.show()

### 2.2 Khám phá Bản chất Từ vựng: Đối lập giữa Thư Ham và Spam
**Mục tiêu:** Trích xuất Top từ khóa trong tiêu đề để phát hiện bản chất thực tế của từng nhóm email.

In [ ]:
# Trích xuất Top 10 từ khóa tiêu đề của Ham và Spam qua data.scripts.eda
keywords_comparison = get_top_keywords_comparison(df, top_n=10)
display(keywords_comparison)

### 2.3 Điều tra Hành vi Ngụy trang & Chiến dịch Spam Hàng loạt
**Mục tiêu:** Điều tra các chiến dịch gửi email lặp lại (`sender` + `subject`) và dấu hiệu hội thoại (`Re:`).

In [ ]:
# Điều tra hành vi phát tán lặp lại và tiền tố trao đổi Re: qua data.scripts.eda
re_dup_res = analyze_re_prefix_and_duplicates(df)
dup_ham = re_dup_res['dup_ham']
dup_spam = re_dup_res['dup_spam']

# subj_lower
# Dùng r'\bre:' vì dấu ':' là ký tự phi từ ngữ (\W), \b đứng sau nó sẽ không khớp nếu theo sau là khoảng trắng
# has_re = subj_lower.str.contains(r'\bre:')
re_ham = re_dup_res['re_ham_ratio']
re_spam = re_dup_res['re_spam_ratio']

print(f"1. Trùng lặp cặp (sender, subject): {re_dup_res['n_duplicates']:,} mẫu (Ham: {re_dup_res['dup_ham']:,} [{re_dup_res['dup_ham']/re_dup_res['n_duplicates']*100:.1f}%] vs Spam: {re_dup_res['dup_spam']:,} [{re_dup_res['dup_spam']/re_dup_res['n_duplicates']*100:.1f}%])")
print(f"2. Tỷ lệ mang tiền tố đối thoại 'Re:': Thư thường = {re_dup_res['re_ham_ratio']:.1f}% vs Thư rác = {re_dup_res['re_spam_ratio']:.1f}%")

### 2.4 Kiểm tra Ngoại lai & Tương quan URL
**Mục tiêu:** Kiểm tra ngoại lai theo IQR và phân tích hiện tượng email siêu dài.

In [ ]:
# Đánh giá ngoại lai IQR qua data.scripts.eda và tương quan URL
iqr_res = detect_iqr_outliers(df['body_len'])
q1, q3, iqr = iqr_res['q1'], iqr_res['q3'], iqr_res['iqr']
iqr_outliers = iqr_res['n_outliers']
print(f"Email ngoại lai siêu dài (> {iqr_res['upper_bound']:.0f} ký tự): {iqr_res['n_outliers']:,} ({iqr_res['outlier_ratio']:.2f}%)")

# Tỷ lệ Spam theo URL
url_rate = df.groupby('urls')['label'].mean() * 100
print(f"Tỷ lệ Spam: Không có URL={url_rate[0]:.2f}%, Có URL={url_rate[1]:.2f}% (Chênh lệch: {url_rate[1]-url_rate[0]:.2f}%)")

### 2.5 Tổng kết EDA & Định hướng Hành động Kỹ thuật (Actionable Insights)

**Mục tiêu:** Tổng hợp toàn bộ các phát hiện định lượng từ khâu EDA để thiết lập kế hoạch tiền xử lý dữ liệu và trích xuất đặc trưng tại Notebook 2.

* **Nguyên tắc phân định trách nhiệm:** Toàn bộ quá trình EDA tại Notebook 1 tuân thủ nghiêm ngặt nguyên tắc **Chỉ Đọc (Read-Only)**, không thay đổi cấu trúc bảng thô và không lưu file dữ liệu mới.
* **Chuyển giao tiền xử lý:** Các công việc làm sạch văn bản NLP (lột HTML/URL, stopwords), khử trùng lặp nội dung sạch (Zero Duplicate Leakage) và trích xuất 7 đặc trưng miền sẽ được thực thi tập trung ở đầu **Notebook 02: Data Preprocessing & Feature Engineering**.

In [ ]:
# Bảng đúc kết EDA định hướng hành động
eda_summary = pd.DataFrame([
    {"Phát hiện EDA": "28 email khuyết subject", "Hành động kỹ thuật": "Điền chuỗi rỗng ''"},
    {"Phát hiện EDA": "Spammer ngụy trang HTML/URL che giấu thư rác gửi loạt", "Hành động kỹ thuật": "Làm sạch NLP rồi khử trùng text_clean"},
    {"Phát hiện EDA": "Từ vựng phân tách rõ (python/bug vs replica/cnn)", "Hành động kỹ thuật": "Dùng TF-IDF n-gram (1,2) + Chi-Square"},
    {"Phát hiện EDA": "Độ dài lệch phải nặng (14.36% ngoại lai)", "Hành động kỹ thuật": "Nén log1p và chuẩn hóa L2"}
])
display(eda_summary)